# Part 2: First-Order ECN Model with Transient Behaviour & Current Dependence

Builds a first-order equivalent circuit network (ECN) model:
- Series resistance R0 + parallel R1-C1 pair
- R0, C1 constant; R1 depends on current magnitude
- Parameterised from `Model_Training_Data_20.csv` (pulse data at 20 °C)
- Validated against `Battery_Testing_Data.csv`

**Sign convention**: negative current = discharge (removes charge from cell).

## Imports and Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.figsize': (12, 5)})

In [2]:
#cell spec (Samsung INR18650-25R datasheet)
Q_nom = 2.5          # nominal capacity [Ah]
V_max = 4.20         # charge cutoff [V]
V_min = 2.50         # discharge cutoff [V]

# load SOC-OCV lookup
df_ocv_raw = pd.read_csv('data/Project 3 data/SOC_OCV.csv')
# file has tab-delimited SOC and OCV inside a single column
soc_ocv = df_ocv_raw.iloc[:, 0].str.split('\t', expand=True).astype(float)
soc_ocv.columns = ['SOC', 'OCV']

# interpolators
ocv_from_soc = interp1d(soc_ocv['SOC'].values, soc_ocv['OCV'].values,
                        kind='linear', fill_value='extrapolate')
soc_from_ocv = interp1d(soc_ocv['OCV'].values, soc_ocv['SOC'].values,
                        kind='linear', fill_value='extrapolate')

# load training data at 20 °C
df_train = pd.read_csv('data/Project 3 data/Model_Training_Data_0.csv')
t_train = df_train['Time (s)'].values
I_train = df_train['Current (A)'].values   # negative = discharge
V_train = df_train['Voltage (V)'].values

# load validation (drive-cycle) data
df_val = pd.read_csv('data/Project 3 data/Battery_Testing_Data.csv').dropna()
t_val = df_val['Time (s)'].values
I_val = df_val['Current (mA)'].values / 1000.0   # mA -> A
V_val = df_val['Voltage (V)'].values
T_val = df_val['Temperature'].values

print(f'Training points : {len(t_train)}')
print(f'Validation points: {len(t_val)}')
print(f'SOC-OCV points  : {len(soc_ocv)}')

FileNotFoundError: [Errno 2] No such file or directory: 'data/Project 3 data/SOC_OCV.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Parametrisation a) — Identify Individual Pulses and Fit ECN Model

At each SOC level the training data contains:
- One long low-current discharge (~0.5 A, C/5 rate) to reach target SOC
- 4 **discharge** test pulses (10 s each at −2.5, −5, −10, −20 A)
- 4 **charge-back** segments after each discharge pulse, at lower current (+1.25, +2.5, +4 A) with durations chosen so the returned charge matches the removed charge

We detect pulse boundaries using the **current gradient** $\mathrm{d}I/\mathrm{d}t$. A sharp spike in $|\mathrm{d}I/\mathrm{d}t|$ indicates a current transition (pulse start or end). This is more robust than a fixed current threshold because it detects *transitions* rather than requiring a magnitude cut-off.

Steps:
1. Compute $\mathrm{d}I/\mathrm{d}t$ at each sample
2. Identify ON edges ($|I|$ increases sharply) and OFF edges ($|I|$ decreases sharply)
3. Pair consecutive ON/OFF edges to delimit each pulse
4. Verify all 64 events (32 discharge + 32 charge-back); then **keep only the 32 discharge pulses** (10 s each) for parameterisation — the charge-back segments are not test pulses and their pre-pulse voltage is not a rested OCV

The histogram below confirms the current-magnitude clusters in the data.

In [ ]:
# ---------- dI/dt histogram to justify gradient-based detection ----------
I_abs_all = np.abs(I_train)

# compute |dI/dt|
dt_arr = np.diff(t_train)
dt_arr[dt_arr == 0] = 1.0
dI_dt = np.abs(np.diff(I_train) / dt_arr)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: |I| histogram (log y) — shows current clusters
axes[0].hist(I_abs_all, bins=300, color='steelblue', edgecolor='k', linewidth=0.3)
axes[0].set_yscale('log')
axes[0].set_xlabel('|Current| [A]'); axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Current magnitude distribution')
axes[0].grid(True, alpha=0.3)

# Panel 2: zoomed |I| showing the gap
mask = (I_abs_all > 0.3) & (I_abs_all < 1.5)
axes[1].hist(I_abs_all[mask], bins=120, color='steelblue', edgecolor='k', linewidth=0.3)
axes[1].axvspan(0.8, 1.25, alpha=0.15, color='green', label='Clean gap')
axes[1].set_xlabel('|Current| [A]'); axes[1].set_ylabel('Count')
axes[1].set_title('Zoomed: SOC-stepping vs test pulses')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Panel 3: |dI/dt| histogram — shows gradient threshold
dI_nonzero = dI_dt[dI_dt > 0.01]  # exclude zero-change samples
axes[2].hist(dI_nonzero, bins=300, color='coral', edgecolor='k', linewidth=0.3)
axes[2].set_yscale('log')
axes[2].axvline(1.0, color='red', ls='--', lw=2, label='Gradient threshold = 1.0 A/s')
axes[2].set_xlabel('|dI/dt| [A/s]'); axes[2].set_ylabel('Count (log scale)')
axes[2].set_title('Current gradient distribution')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Transitions with |dI/dt| > 1.0 A/s: {np.sum(dI_dt > 1.0)}')
print(f'Transitions with |dI/dt| > 0.5 A/s: {np.sum(dI_dt > 0.5)}')

In [ ]:
def find_pulses_didt(time, current, voltage, grad_thresh=1.0):
    """
    Detect current pulses via the gradient dI/dt.

    A pulse START is where |dI/dt| exceeds the threshold and |I| increases.
    A pulse END   is where |dI/dt| exceeds the threshold and |I| decreases.
    Consecutive start/end pairs delimit individual pulses.

    Parameters
    ----------
    grad_thresh : float
        Minimum |dI/dt| [A/s] to register a current transition.
        1.0 A/s sits between SOC-stepping transitions (~0.5 A/s)
        and the smallest test-pulse transition (1.25 A/s).
    """
    dt = np.diff(time)
    dt[dt == 0] = 1.0
    dI_dt = np.abs(np.diff(current) / dt)

    # classify each large-gradient sample as ON or OFF edge
    trans_idx = np.where(dI_dt > grad_thresh)[0]
    starts, ends = [], []
    for idx in trans_idx:
        if abs(current[idx + 1]) > abs(current[idx]):
            starts.append(idx + 1)   # first sample inside the pulse
        else:
            ends.append(idx + 1)     # first sample after the pulse

    # pair each start with the next end
    pulses = []
    ei = 0
    for s in starts:
        while ei < len(ends) and ends[ei] <= s:
            ei += 1
        if ei >= len(ends):
            break
        e = ends[ei]
        dur = time[e] - time[s]
        # rest voltage = mean of 10 samples before pulse
        v_before = np.mean(voltage[max(0, s - 10):s])
        pulses.append({
            'idx_start': s, 'idx_end': e,
            't_start': time[s], 't_end': time[e],
            'duration': dur,
            'I_avg': np.mean(current[s:e]),
            'V_before': v_before,       # OCV estimate (rest voltage)
            'V_start': voltage[s],      # first voltage sample during pulse
        })
        ei += 1

    return pulses

pulses = find_pulses_didt(t_train, I_train, V_train)
print(f'Detected {len(pulses)} pulses via dI/dt')

dis_pulses = [p for p in pulses if p['I_avg'] < 0]
chg_pulses = [p for p in pulses if p['I_avg'] > 0]
print(f'  {len(dis_pulses)} discharge (10 s each)')
print(f'  {len(chg_pulses)} charge-back (duration matches removed charge)')

for p in pulses[:8]:
    ptype = 'DIS' if p['I_avg'] < 0 else 'CHG'
    print(f"  [{ptype}] t={p['t_start']:.0f}s  dur={p['duration']:.1f}s  "
          f"I={p['I_avg']:.2f}A  V_before={p['V_before']:.4f}V  V_start={p['V_start']:.4f}V")

In [ ]:
# ---------- verify pulse structure: 4 discharge + 4 charge per SOC ----------
for p in pulses:
    p['SOC_est'] = float(soc_from_ocv(p['V_before']))

soc_levels = sorted(set(round(p['SOC_est'] / 10) * 10 for p in pulses))
for p in pulses:
    p['SOC_group'] = min(soc_levels, key=lambda s: abs(p['SOC_est'] - s))

print(f'{"SOC level":<12} {"Discharge":<12} {"Charge":<12} {"Total":<8} {"Discharge currents [A]":<35} {"Charge currents [A]"}')
print('-' * 115)

all_ok = True
for s in soc_levels:
    group = [p for p in pulses if p['SOC_group'] == s]
    dis = [p for p in group if p['I_avg'] < 0]
    chg = [p for p in group if p['I_avg'] > 0]
    dis_I = sorted([round(p['I_avg'], 1) for p in dis])
    chg_I = sorted([round(p['I_avg'], 1) for p in chg])
    ok = len(dis) == 4 and len(chg) == 4
    status = 'OK' if ok else 'MISMATCH'
    if not ok:
        all_ok = False
    print(f'SOC~{s:3d}%     {len(dis):<12d} {len(chg):<12d} {len(group):<8d} {str(dis_I):<35s} {str(chg_I)}  {status}')

total_dis = sum(1 for p in pulses if p['I_avg'] < 0)
total_chg = sum(1 for p in pulses if p['I_avg'] > 0)
print(f'\nTotal: {total_dis} discharge + {total_chg} charge = {len(pulses)} pulses')
print(f'Expected: {4*len(soc_levels)} discharge + {4*len(soc_levels)} charge = {8*len(soc_levels)} total')

# hard check — fail loudly if structure is wrong
assert len(pulses) == 8 * len(soc_levels), \
    f'Expected {8*len(soc_levels)} pulses, got {len(pulses)}'
assert all_ok, 'Not every SOC level has exactly 4 discharge + 4 charge pulses!'
print('\n==> CHECK PASSED: every SOC level has exactly 4 discharge + 4 charge pulses')

# verify discharge pulses are all 10 s (per brief)
dis_durs = [p['duration'] for p in pulses if p['I_avg'] < 0]
chg_durs = [p['duration'] for p in pulses if p['I_avg'] > 0]
print(f'\nDischarge durations: {sorted(set(round(d,1) for d in dis_durs))} s  (all 10 s per brief)')
print(f'Charge-back durations: {sorted(set(round(d,1) for d in chg_durs))} s  (match removed charge)')

print(f'\nDiscovered discharge currents: {sorted(set(round(p["I_avg"],1) for p in pulses if p["I_avg"]<0))} A')
print(f'Discovered charge currents:    {sorted(set(round(p["I_avg"],1) for p in pulses if p["I_avg"]>0))} A')

# ── FILTER: keep only 10 s discharge pulses for parameterisation ──
# The charge-back segments are NOT test pulses — they simply return charge
# to hold the cell at the same SOC. Their V_before is not a rest OCV
# (the cell hasn't equilibrated after the discharge), so including them
# would corrupt R0, R1, C1 estimates.
pulses = [p for p in pulses if p['I_avg'] < 0]
print(f'\n>>> Filtered to {len(pulses)} discharge pulses (10 s each) for parameterisation')

### Assign SOC to Each Pulse

Estimate SOC from rest voltage before each pulse using the OCV curve.

In [ ]:
# SOC assignment already done in verification cell above
# just confirm grouping
soc_levels = sorted(set(round(p['SOC_est'] / 10) * 10 for p in pulses))
print(f'SOC levels: {soc_levels}')
for s in soc_levels:
    # use nearest-level assignment (no overlap)
    n = sum(1 for p in pulses if p['SOC_group'] == s)
    print(f'  SOC~{s}%: {n} pulses')

### Extract R0 from Instantaneous Voltage Step

When current steps from 0 to $I$, the terminal voltage instantly jumps by $I \cdot R_0$:

$$V_{\text{start}} = V_{\text{rest}} + I \cdot R_0$$

So:

$$R_0 = \frac{V_{\text{start}} - V_{\text{rest}}}{I}$$

For discharge ($I < 0$): $V_{\text{start}} < V_{\text{rest}}$ and $I < 0$, giving $R_0 > 0$.

In [ ]:
for p in pulses:
    if abs(p['I_avg']) > 0.2:
        # R0 = (V_start - V_rest) / I
        p['R0_inst'] = (p['V_start'] - p['V_before']) / p['I_avg']
    else:
        p['R0_inst'] = np.nan

# collect valid R0 values (should be positive)
R0_list = [p['R0_inst'] for p in pulses
           if not np.isnan(p['R0_inst']) and p['R0_inst'] > 0]

R0_avg = np.mean(R0_list)

print(f'R0 from {len(R0_list)} pulses:')
print(f'  mean   = {R0_avg*1e3:.2f} mOhm')
print(f'  median = {np.median(R0_list)*1e3:.2f} mOhm')
print(f'  std    = {np.std(R0_list)*1e3:.2f} mOhm')
print(f'  range  = [{min(R0_list)*1e3:.2f}, {max(R0_list)*1e3:.2f}] mOhm')

In [ ]:
# visualise R0 vs SOC and current
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

R0_data = [(p['SOC_est'], p['I_avg'], p['R0_inst'])
           for p in pulses if not np.isnan(p.get('R0_inst', np.nan)) and p['R0_inst'] > 0]
socs, currents, r0s = zip(*R0_data)

sc = axes[0].scatter(socs, np.array(r0s)*1e3, c=currents, cmap='coolwarm', edgecolors='k', s=50)
axes[0].set_xlabel('SOC [%]'); axes[0].set_ylabel('R0 [mOhm]')
axes[0].set_title('R0 vs SOC (coloured by current)')
axes[0].axhline(R0_avg*1e3, color='gray', ls='--', label=f'mean={R0_avg*1e3:.1f} mOhm')
axes[0].legend(); plt.colorbar(sc, ax=axes[0], label='Current [A]')

axes[1].scatter(np.abs(currents), np.array(r0s)*1e3, c=socs, cmap='viridis', edgecolors='k', s=50)
axes[1].set_xlabel('|Current| [A]'); axes[1].set_ylabel('R0 [mOhm]')
axes[1].set_title('R0 vs |Current| (coloured by SOC)')
axes[1].axhline(R0_avg*1e3, color='gray', ls='--')

plt.tight_layout(); plt.show()

### Fit R1 and τ with R0 Fixed

Model equation during a constant-current pulse:

$$V(t) = \text{OCV} + I \cdot R_0 + I \cdot R_1 \left(1 - e^{-t/\tau}\right)$$

With $R_0$ known, fit $R_1$ and $\tau = R_1 C_1$ for each pulse.

In [ ]:
def fit_R1_tau(time_arr, volt_arr, I_pulse, V_ocv, R0):
    """Fit R1 and tau to a single pulse with R0 fixed."""
    t_rel = time_arr - time_arr[0]

    def model(t, R1, tau):
        return V_ocv + I_pulse * R0 + I_pulse * R1 * (1 - np.exp(-t / tau))

    try:
        popt, _ = curve_fit(model, t_rel, volt_arr, p0=[0.01, 3.0],
                            bounds=([1e-5, 0.1], [0.5, 200]), maxfev=10000)
        R1, tau = popt
        C1 = tau / R1
        return {'R1': R1, 'C1': C1, 'tau': tau}
    except (RuntimeError, ValueError):
        return None

# fit all pulses
fit_results = []
for p in pulses:
    if abs(p['I_avg']) < 0.2:
        continue
    i0, i1 = p['idx_start'], p['idx_end']
    result = fit_R1_tau(t_train[i0:i1], V_train[i0:i1],
                        p['I_avg'], p['V_before'], R0_avg)
    if result:
        result['SOC'] = p['SOC_est']
        result['I'] = p['I_avg']
        fit_results.append(result)

df_fits = pd.DataFrame(fit_results)
print(f'Fitted {len(df_fits)} pulses')
print(f'R1: {df_fits["R1"].min()*1e3:.2f} – {df_fits["R1"].max()*1e3:.2f} mOhm')
print(f'C1: {df_fits["C1"].min():.0f} – {df_fits["C1"].max():.0f} F')
print(f'tau: {df_fits["tau"].min():.1f} – {df_fits["tau"].max():.1f} s')

## Parametrisation b) — Parameter Table: R0, R1, C1 vs SOC and Current

In [ ]:
import os

# -- CHECK: Make sure the data actually exists --
if 'df_fits' not in locals() or df_fits.empty:
    print("❌ ERROR: df_fits is empty or doesn't exist! Make sure you go up and click 'Run All' first.")
else:
    print(f"✅ df_fits has {len(df_fits)} rows. Proceeding to save...")

    # 1. Define the folder and make sure it exists
    folder_name = 'extracted_parameters'
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    # 2. Add the temperature tracking column
    df_fits['Temperature_C'] = 0

    # 3. Construct the path and save the CSV
    file_path = os.path.join(folder_name, 'R1_values_by_SOC_and_Current_0_deg.csv')
    df_fits.to_csv(file_path, index=False)

    print(f"Data successfully saved to Colab directory: {file_path}")
    print("Triggering download to your computer...")

    # 4. Use Colab's specific download module
    from google.colab import files
    files.download(file_path)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, label in zip(axes,
                          ['R1', 'C1', 'tau'],
                          ['R1 [mOhm]', 'C1 [F]', 'tau [s]']):
    scale = 1e3 if col == 'R1' else 1
    sc = ax.scatter(df_fits['SOC'], df_fits[col]*scale, c=df_fits['I'],
                    cmap='coolwarm', edgecolors='k', s=50)
    ax.set_xlabel('SOC [%]'); ax.set_ylabel(label)
    ax.set_title(f'{col} vs SOC')
    plt.colorbar(sc, ax=ax, label='Current [A]')

plt.tight_layout(); plt.show()

## Parametrisation c–f) — Average R0 and C1

Per the brief: take the average values for R0 and C1 across all SOCs and currents, then re-fit R1 with both held fixed.

In [ ]:
C1_avg = df_fits['C1'].mean()

print(f'C1 average = {C1_avg:.1f} F')
print(f'C1 std     = {df_fits["C1"].std():.1f} F  ({df_fits["C1"].std()/C1_avg*100:.0f}% relative)')

### Re-fit R1 with R0 and C1 Fixed

In [ ]:
def refit_R1_only(time_arr, volt_arr, I_pulse, V_ocv, R0, C1):
    """Fit only R1 with R0 and C1 held constant."""
    t_rel = time_arr - time_arr[0]

    def model(t, R1):
        tau = R1 * C1
        return V_ocv + I_pulse * R0 + I_pulse * R1 * (1 - np.exp(-t / tau))

    try:
        popt, _ = curve_fit(model, t_rel, volt_arr, p0=[0.01],
                            bounds=([1e-5], [1.0]), maxfev=10000)
        return popt[0]
    except (RuntimeError, ValueError):
        return None

refit_results = []
for p in pulses:
    if abs(p['I_avg']) < 0.2:
        continue
    i0, i1 = p['idx_start'], p['idx_end']
    R1_new = refit_R1_only(t_train[i0:i1], V_train[i0:i1],
                           p['I_avg'], p['V_before'], R0_avg, C1_avg)
    if R1_new is not None:
        refit_results.append({
            'SOC': p['SOC_est'],
            'I': p['I_avg'],
            'R1': R1_new
        })

df_R1 = pd.DataFrame(refit_results)
print(f'Re-fitted R1 for {len(df_R1)} pulses')
print(df_R1.round(5).to_string())

## Parametrisation i–j) — R1 Current Dependence: Exponential Fit

### i) Why an exponential function?
Charge-transfer kinetics are governed by the Butler-Volmer equation, which has exponential form. At higher currents the reaction overpotential per unit current decreases, so the effective resistance drops exponentially with |I|.

### j) Two candidate functions

**Option A** (simple exponential): $R_1(I) = a \cdot e^{-b|I|}$

**Option B** (shifted exponential): $R_1(I) = a \cdot e^{-b|I|} + c$

Option B prevents R1 dropping to zero at large currents, which is physically realistic (some polarisation resistance always remains).

In [ ]:
# fit on discharge pulses (I < 0)
df_R1_dis = df_R1[df_R1['I'] < -0.2].copy()
df_R1_chg = df_R1[df_R1['I'] > 0.2].copy()

I_abs = np.abs(df_R1_dis['I'].values)
R1_vals = df_R1_dis['R1'].values

# Option A: R1 = a * exp(-b * |I|)
def option_A(I, a, b):
    return a * np.exp(-b * I)

# Option B: R1 = a * exp(-b * |I|) + c
def option_B(I, a, b, c):
    return a * np.exp(-b * I) + c

popt_A, _ = curve_fit(option_A, I_abs, R1_vals, p0=[0.05, 0.1], maxfev=10000)
popt_B, _ = curve_fit(option_B, I_abs, R1_vals, p0=[0.05, 0.1, 0.005],
                      bounds=([0, 0, 0], [1, 10, 0.5]), maxfev=10000)

res_A = np.sqrt(np.mean((option_A(I_abs, *popt_A) - R1_vals)**2))
res_B = np.sqrt(np.mean((option_B(I_abs, *popt_B) - R1_vals)**2))

print(f'Option A: a={popt_A[0]*1e3:.3f} mOhm, b={popt_A[1]:.4f} /A  RMSE={res_A*1e3:.3f} mOhm')
print(f'Option B: a={popt_B[0]*1e3:.3f} mOhm, b={popt_B[1]:.4f} /A, c={popt_B[2]*1e3:.3f} mOhm  RMSE={res_B*1e3:.3f} mOhm')

# plot
I_plot = np.linspace(0.5, 22, 200)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(I_abs, R1_vals*1e3, c='tab:blue', label='Discharge', zorder=3)
ax.scatter(np.abs(df_R1_chg['I']), df_R1_chg['R1']*1e3, c='tab:red',
           marker='^', label='Charge', zorder=3)
ax.plot(I_plot, option_A(I_plot, *popt_A)*1e3, 'b--',
        label=f'A: RMSE={res_A*1e3:.2f} mOhm')
ax.plot(I_plot, option_B(I_plot, *popt_B)*1e3, 'r-',
        label=f'B: RMSE={res_B*1e3:.2f} mOhm')
ax.set_xlabel('|Current| [A]'); ax.set_ylabel('R1 [mOhm]')
ax.set_title('R1 vs |Current| — discharge data used for fit')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- select best fit (Option B preferred — see discussion below) ---
R1_params = popt_B

def R1_of_I(I_current):
    """R1 [Ohm] as a function of current [A]. Uses |I|."""
    a, b, c = R1_params
    return a * np.exp(-b * np.abs(I_current)) + c

print('Selected: Option B (shifted exponential)')
print(f'  R1(I) = {R1_params[0]*1e3:.3f}e-3 * exp(-{R1_params[1]:.4f}*|I|) + {R1_params[2]*1e3:.3f}e-3  [Ohm]')

### Parameter Summary

| Parameter | Value | Notes |
|-----------|-------|-------|
| R0 | (see above) | constant, from instantaneous voltage step |
| C1 | (see above) | constant, averaged across all pulses |
| R1(I) | Option B fit | depends on current magnitude |

In [ ]:
print('=== Final Model Parameters ===')
print(f'R0 = {R0_avg*1e3:.2f} mOhm  (constant)')
print(f'C1 = {C1_avg:.1f} F  (constant)')
print(f'R1(I) = {R1_params[0]*1e3:.3f}e-3 * exp(-{R1_params[1]:.4f}*|I|) + {R1_params[2]*1e3:.3f}e-3  [Ohm]')
print(f'\nPlaceholder values from brief: R0=40 mOhm, R1=5 mOhm, C1=5000 F')

## Implementation — ECN Simulation Function

Discrete-time stepping:

1. **Coulomb counting**: $\text{SOC}_{k+1} = \text{SOC}_k + \frac{I_k \cdot \Delta t}{Q_{\text{nom}} \cdot 3600} \times 100$
2. **OCV lookup**: $\text{OCV}_k = f(\text{SOC}_k)$
3. **RC update**: $V_{RC,k+1} = V_{RC,k} \cdot e^{-\Delta t/\tau_k} + I_k \cdot R_{1,k}\,(1 - e^{-\Delta t/\tau_k})$
4. **Terminal voltage**: $V_k = \text{OCV}_k + I_k \cdot R_0 + V_{RC,k}$

Note: with negative $I$ for discharge, all voltage drops are negative (voltage decreases).

In [ ]:
def simulate_ecn(time, current, SOC_init, R0, C1, R1_func, Q_nom, ocv_interp):
    """
    Simulate first-order ECN.
    Sign convention: negative current = discharge.
    V = OCV + I*R0 + V_rc   (all drops are via the sign of I)
    """
    n = len(time)
    V_pred = np.zeros(n)
    SOC    = np.zeros(n)
    V_rc   = 0.0
    SOC[0] = SOC_init

    for k in range(n):
        I_k = current[k]
        OCV_k = float(ocv_interp(np.clip(SOC[k], 0, 100)))

        # current-dependent R1 and time constant
        R1_k = R1_func(I_k)
        tau_k = R1_k * C1

        # terminal voltage
        V_pred[k] = OCV_k + I_k * R0 + V_rc

        if k < n - 1:
            dt = time[k+1] - time[k]
            if dt > 0 and tau_k > 0:
                exp_term = np.exp(-dt / tau_k)
                # RC element charges towards I*R1
                V_rc = V_rc * exp_term + I_k * R1_k * (1 - exp_term)
            # coulomb counting
            SOC[k+1] = SOC[k] + (I_k * dt) / (Q_nom * 3600) * 100
            SOC[k+1] = np.clip(SOC[k+1], 0, 100)

    return V_pred, SOC

## Implementation — Training Data Validation

In [ ]:
# pick a representative discharge pulse (~-10 A)
test_pulse = [p for p in pulses if round(p['I_avg']) == -10][0]
i0, i1 = test_pulse['idx_start'], test_pulse['idx_end']

margin = 100
i0e = max(0, i0 - margin)
i1e = min(len(t_train), i1 + margin)

V_sim, _ = simulate_ecn(
    t_train[i0e:i1e], I_train[i0e:i1e],
    test_pulse['SOC_est'], R0_avg, C1_avg, R1_of_I, Q_nom, ocv_from_soc)

t_seg = t_train[i0e:i1e] - t_train[i0e]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_seg, V_train[i0e:i1e], 'k-', lw=2, label='Measured')
ax.plot(t_seg, V_sim, 'r--', lw=2, label='ECN model')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Voltage [V]')
ax.set_title(f'Single pulse check  (I~{test_pulse["I_avg"]:.0f} A, SOC~{test_pulse["SOC_est"]:.0f}%)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Implementation — Drive-Cycle Validation

Find the SOC just before the discharge begins so the model voltage matches the measured rest voltage.

In [ ]:
# find where the drive-cycle discharge starts
discharge_idx = None
for i in range(100, len(I_val)):
    if I_val[i] < -0.5:
        discharge_idx = i
        break

V_rest = np.mean(V_val[discharge_idx-10 : discharge_idx])
SOC_init = float(soc_from_ocv(V_rest))

print(f'Discharge starts at index {discharge_idx}, t = {t_val[discharge_idx]:.0f} s')
print(f'Rest voltage = {V_rest:.4f} V  ->  SOC_init = {SOC_init:.1f}%')

In [ ]:
# run model from discharge start onward
t_dc = t_val[discharge_idx:]
I_dc = I_val[discharge_idx:]
V_dc = V_val[discharge_idx:]

V_pred, SOC_pred = simulate_ecn(
    t_dc, I_dc, SOC_init, R0_avg, C1_avg, R1_of_I, Q_nom, ocv_from_soc)

In [ ]:
t_rel = t_dc - t_dc[0]
error = V_pred - V_dc

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# voltage comparison
axes[0].plot(t_rel, V_dc, 'k-', lw=1, label='Measured')
axes[0].plot(t_rel, V_pred, 'r-', lw=1, alpha=0.8, label='ECN model')
axes[0].set_ylabel('Voltage [V]'); axes[0].legend(loc='upper right')
axes[0].set_title('Part 2: First-Order ECN vs Drive Cycle')
axes[0].grid(True, alpha=0.3)

# error
axes[1].plot(t_rel, error*1e3, 'b-', lw=0.8)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('Error [mV]')
rmse = np.sqrt(np.mean(error**2))*1e3
mae  = np.mean(np.abs(error))*1e3
maxe = np.max(np.abs(error))*1e3
axes[1].set_title(f'RMSE = {rmse:.1f} mV,  MAE = {mae:.1f} mV,  Max = {maxe:.1f} mV')
axes[1].grid(True, alpha=0.3)

# current
axes[2].plot(t_rel, I_dc, 'g-', lw=0.8)
axes[2].set_ylabel('Current [A]'); axes[2].set_xlabel('Time [s]')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'RMSE: {rmse:.2f} mV')
print(f'MAE:  {mae:.2f} mV')
print(f'Max:  {maxe:.2f} mV')

### SOC Trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_rel, SOC_pred, 'b-')
ax.set_xlabel('Time [s]'); ax.set_ylabel('SOC [%]')
ax.set_title('Predicted SOC During Drive Cycle')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()